# Notebook 03 — Clustering & Cell Type Annotation

**Goal:** Identify distinct cell populations in the UMAP and annotate them with biological cell type labels.

## Overview

This is the core biological analysis of a scRNA-seq experiment.  
We use **Leiden algorithm** (graph-based clustering) to group cells with similar transcriptional profiles, then assign biological labels using:

1. **Marker gene expression** (from van Galen 2019, Table S2)
2. **van Galen's own annotations** (available in `.obs['CellType']` from the anno files)
3. **Gene scoring** (automated scoring using `src/utils.py`)

## AML cell types in the hierarchy (van Galen 2019)

```
HSC → Progenitor → GMP → Promono → Mono
                ↘ Erythroid
                ↘ Platelet/MEP
Malignant cells = blocked at one of these stages
```

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from src.utils import (
    run_leiden_clustering,
    score_cell_types,
    assign_cell_type_from_scores,
    AML_MARKER_GENES,
    set_plot_style
)

set_plot_style()
sc.settings.verbosity = 1

In [ ]:
# Load preprocessed data from notebook 02
adata = sc.read('../data/raw/adata_02_preprocessed.h5ad')
print(f"Loaded: {adata.n_obs} cells × {adata.n_vars} genes")
print(f"Columns in .obs: {adata.obs.columns.tolist()}")

## 3.1 — Leiden Clustering

Leiden is a graph-based clustering algorithm:
1. Build KNN graph (already done in notebook 02)
2. Optimize a modularity function to find communities

`resolution` controls granularity:
- Low (0.3): few, broad clusters
- High (1.5): many, fine-grained clusters

For AML cell type deconvolution, 0.4–0.8 is usually appropriate.

In [ ]:
adata = run_leiden_clustering(adata, resolution=0.5)

In [ ]:
# UMAP colored by Leiden cluster
sc.pl.umap(adata, color='leiden', legend_loc='on data', title='Leiden Clusters')
plt.savefig('../figures/03_umap_leiden.png', dpi=150, bbox_inches='tight')

## 3.2 — Van Galen Cell Type Labels

The authors of GSE116256 provided their own cell type annotations.  
These should be in `.obs['CellType']` (loaded from the `.anno.txt.gz` files).  
We compare these expert labels to our clustering.

In [ ]:
# Check if van Galen annotations are present
if 'CellType' in adata.obs.columns:
    print("Van Galen CellType labels found!")
    print(adata.obs['CellType'].value_counts())
else:
    print("CellType column not found — check that anno files were loaded correctly.")
    print(f"Available columns: {adata.obs.columns.tolist()}")

In [ ]:
# If van Galen labels exist, plot them on UMAP
if 'CellType' in adata.obs.columns:
    sc.pl.umap(
        adata,
        color='CellType',
        title='Van Galen 2019 — Expert Cell Type Labels',
        legend_loc='right margin'
    )
    plt.savefig('../figures/03_umap_van_galen_labels.png', dpi=150, bbox_inches='tight')

## 3.3 — Marker Gene Visualization

We plot key marker genes for each cell type on the UMAP to validate cluster identities.  
These are the canonical markers from van Galen 2019 Table S2.

In [ ]:
# Key markers per cell type — a subset for visualization
key_markers = {
    'HSC / Progenitor': ['CD34', 'SPINK2'],
    'Myeloid / GMP':    ['MPO', 'ELANE'],
    'Monocyte':         ['CD14', 'LYZ', 'S100A8'],
    'Erythroid':        ['HBB', 'GYPA'],
    'T / NK cell':      ['CD3D', 'NKG7'],
}

for cell_type, genes in key_markers.items():
    # Keep only genes present in the dataset
    available = [g for g in genes if g in adata.var_names]
    if available:
        sc.pl.umap(
            adata, color=available,
            title=[f'{g} ({cell_type})' for g in available],
            use_raw=True, vmax='p99'
        )
        plt.savefig(f'../figures/03_umap_markers_{cell_type.replace(" / ", "_").replace(" ", "_")}.png',
                    dpi=100, bbox_inches='tight')

## 3.4 — Differential Expression per Cluster

Find top marker genes for each Leiden cluster using Wilcoxon rank-sum test.  
This helps manually annotate clusters that don't have pre-existing labels.

In [ ]:
# Compute differentially expressed genes per cluster
sc.tl.rank_genes_groups(adata, groupby='leiden', method='wilcoxon', use_raw=True)
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)
plt.savefig('../figures/03_cluster_marker_genes.png', dpi=150, bbox_inches='tight')

In [ ]:
# Top 5 marker genes per cluster as DataFrame
top_markers = sc.get.rank_genes_groups_df(adata, group=None)
top_markers_per_cluster = (
    top_markers.groupby('group')
    .apply(lambda x: x.nlargest(5, 'scores'))
    .reset_index(drop=True)
)
print(top_markers_per_cluster[['group', 'names', 'scores', 'pvals_adj']].to_string())

## 3.5 — Automated Cell Type Scoring

We use our `score_cell_types()` utility to score each cell against each cell type's marker genes.  
Then assign the highest-scoring label as the predicted cell type.

In [ ]:
adata = score_cell_types(adata)
adata = assign_cell_type_from_scores(adata)

print("Predicted cell type distribution:")
print(adata.obs['predicted_cell_type'].value_counts())

In [ ]:
# Final UMAP with predicted labels
sc.pl.umap(
    adata,
    color='predicted_cell_type',
    title='AML Cell Type Landscape — Predicted Labels',
    legend_loc='right margin',
    frameon=False
)
plt.savefig('../figures/03_umap_predicted_cell_types.png', dpi=150, bbox_inches='tight')

In [ ]:
# Heatmap: top marker genes per predicted cell type
all_markers = [g for genes in AML_MARKER_GENES.values() for g in genes if g in adata.var_names]
sc.pl.heatmap(
    adata,
    var_names=list(dict.fromkeys(all_markers)),  # deduplicate while preserving order
    groupby='predicted_cell_type',
    use_raw=True,
    dendrogram=True,
    figsize=(14, 6)
)
plt.savefig('../figures/03_marker_heatmap.png', dpi=150, bbox_inches='tight')

In [ ]:
# Save annotated object
adata.write('../data/raw/adata_03_annotated.h5ad')
print("Saved: ../data/raw/adata_03_annotated.h5ad")

## Summary

At this stage we have:
- Leiden clusters with marker gene assignments
- Van Galen expert labels (from paper)
- Automated cell type scores

Cross-checking between these three annotation strategies gives biological confidence.

**Next:** Notebook 04 — Pseudotime trajectory analysis (PAGA + DPT).